# Restart capabilities

"Restart" capabilities refer to pywatershed ability to save state and to then use that state as initial conditions in a separate run.

This notebook demonstrates how to:
- Run a model and write restart files at specified intervals
- Use those restart files to initialize subsequent model runs
- Verify that restarted runs produce identical results to continuous runs

Restart capabilities are essential for long simulations, allowing you to break them into smaller chunks or for recovering from interruptions without losing progress. Restart capabilities are also essential for performing external operations on model state at regular intervals (e.g. data assimilation).

In this notebook, we'll demonstrate running the model for just 5 days but we'll do it two ways:
1. As a single run for 5 days.
2. As 5 runs, each a day long.

There is a variety of restart options for both writing and reading. We only show one example usage below, but the following documentation is taken from the [Process class documentation](https://pywatershed.readthedocs.io/en/latest/api/generated/pywatershed.base.Process.html). Restart options are available for all Process subclasses. These can be passed as arguments to individual processes or set in `control.options`:

- **`restart_write`** (`bool` or `pathlib.Path`, default `False`): Same as `restart_read` but for writing. The directory will be created if it doesn't exist.

- **`restart_write_freq`** (`"y"`, `"m"`, `"d"`, `"f"`, or `False`, default `False`): Sets the frequency of restart output. `"y"` for yearly (last day of year), `"m"` for monthly (last day of month), `"d"` for daily, `"f"` for final (only at `control.end_time`). If `restart_write` is not `False` and `restart_write_freq` is `False`, the default of `"f"` is used.

- **`restart_read`** (`bool` or `pathlib.Path`, default `False`): If `False`, `control.options` will be examined for this key. If `True`, the working directory is searched for restart files. If a `pathlib.Path`, this specifies an alternative directory to search for restart files. Files searched for are of the pattern `YYYY-mm-dd-varname.nc` where the date is `control.init_time`.

Please note that this notebook relies on notebook `03_compare_pws_prms.ipynb` having already run. If not, there will be an input error when running the pywatershed model below. 

In [ ]:
import pathlib as pl
from pprint import pprint
import shutil

import jupyter_black
import numpy as np
import pywatershed as pws
import xarray as xr

jupyter_black.load()

## Setup

We initialize the model on 1978-12-31 (with default/cold state) and run days 1979-01-01 through 1979-01-05. 

In [ ]:
input_dir = pl.Path("../test_data/drb_2yr/output/")
domain_dir = pws.constants.__pywatershed_root__ / "data/drb_2yr"

t0 = np.datetime64("1978-12-31T00:00:00")
timestep = np.timedelta64(24, "h")
n_steps = 5

nb_output_dir = pl.Path("./08_restart_streamflow")

## Configure control and parameters

Helper functions to create control and parameter objects. The control object is configured to:
- Write restart files at daily frequency (`restart_write_freq = "d"`)
- Optionally read from a restart directory to initialize model state

In [ ]:
def get_control(init_time, end_time, restart_read: pl.Path = False):
    control = pws.Control.load_prms(
        domain_dir / "nhm.control", warn_unused_options=False
    )
    control.options["input_dir"] = input_dir
    control.options["restart_write"] = restart_dir
    control.options["restart_write_freq"] = "d"
    if restart_read:
        control.options["restart_read"] = restart_read
    del control.options["netcdf_output_dir"]
    del control.options["netcdf_output_var_names"]
    control.edit_end_time(end_time)
    control.edit_init_start_times(init_time)
    return control

In [ ]:
def get_params():
    return pws.parameters.PrmsParameters.load(domain_dir / "myparam.param")

## Define model processes

We'll use a simple 2-process model with groundwater and streamflow routing to demonstrate restart functionality. However, all pywatershed processes are supported. 

In [ ]:
nhm_processes = [
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

## Single run for 5 days

Run the full 5-day period in a single continuous simulation. The restart files written at the end of the run will serve as the "truth" to verify that the restarted runs produce identical results. To only get the restart at the end of the runs, we'll override the `restart_write_freq` option as `'f'` (instead of `'d'` as set in the function above).

In [ ]:
restart_dir = nb_output_dir / "restart_single_run"
if restart_dir.exists():
    shutil.rmtree(restart_dir)

control = get_control(t0, t0 + timestep * n_steps)
control.options["restart_write_freq"] = "f"

nhm = pws.Model(
    nhm_processes,
    control=control,
    parameters=get_params(),
)
nhm.run(finalize=True)

pprint(sorted(restart_dir.glob("*.nc")))

## Five runs of 1-day each

Running the model for one day creates the first restart file that will be used to initialize the next run.

In [ ]:
restart_dir = nb_output_dir / "restart_multi_run"
if restart_dir.exists():
    shutil.rmtree(restart_dir)

nhm = pws.Model(
    nhm_processes,
    control=get_control(t0, t0 + timestep),
    parameters=get_params(),
)
nhm.run(finalize=True)
pprint(sorted(restart_dir.glob("*.nc")))

Runs 2-5: Continue with restart read/write

Run the remaining timesteps sequentially. Each run:
1. Reads state from the previous restart file
2. Advances one timestep
3. Writes a new restart file

This simulates breaking a longer run into multiple shorter runs.

In [ ]:
for ii in range(n_steps - 1):
    init_time = t0 + (ii + 1) * timestep
    nhm = pws.Model(
        nhm_processes,
        control=get_control(
            init_time, init_time + timestep, restart_read=restart_dir
        ),
        parameters=get_params(),
    )
    nhm.run(finalize=True)
    print(f"{ii=}: {init_time=}")

pprint(sorted(restart_dir.glob("*.nc")))

## Verify restart accuracy

Compare the final state from both approaches. The restarted runs should produce exactly the same results as the continuous run, confirming that restart functionality preserves model state perfectly.

In [ ]:
final_time_stamp = (t0 + n_steps * timestep).item().strftime("%Y-%m-%d")
for no_rs_file in sorted(restart_dir.glob(f"{final_time_stamp}*")):
    rs_file = nb_output_dir / f"restart_single_run/{no_rs_file.name}"
    print(no_rs_file)
    print(rs_file)
    no_rs_da = xr.load_dataarray(no_rs_file)
    rs_da = xr.load_dataarray(rs_file)
    # xr.testing.assert_allclose(no_rs_da, rs_da)
    xr.testing.assert_equal(no_rs_da, rs_da)
    print(no_rs_file.name, "passes")